# Resultados CEC 2022: parrilla principal + algoritmos de referencia

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexanderefe/test_transfer_in_bioalgorithms/blob/main/notebooks/analisis_resultados.ipynb)

Visualización de los resultados del middleware de transferencia de conocimiento
sobre CEC 2022 (12 funciones, D = 10 y 20, MaxFES 5·10⁴ … 5·10⁷, 51 semillas).

| Conjunto | Algoritmos | Corridas | Datos |
|---|---|---|---|
| `principal+referencia` | Propuesto, PSO, DE, IPOP-CMA-ES, MadDE, NL-SHADE-RSP + GA, JADE, CLPSO, ABC, CLONALG, opt-aiNet | 58 752 | `Resultados_Union_Referencia/` |
| `principal` | Los 6 primeros | 29 376 | `Resultados_Union/` |

**Uso:** *Entorno de ejecución → Ejecutar todas*. La primera vez clona el
repositorio y regenera tablas y figuras (unos minutos). Después, los selectores
de la sección 4 permiten recorrer todas las figuras sin volver a calcular nada.
Para cambiar de conjunto: editar `CONJUNTO` (sección 2) y *Entorno de ejecución →
Reiniciar sesión y ejecutar todo*.

Cómo leer cada tabla y figura: `LEEME.md` de la carpeta de análisis (se muestra
al final).

## 1. Entorno

In [ ]:
import os, sys

REPO_URL = "https://github.com/alexanderefe/test_transfer_in_bioalgorithms.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not os.path.isdir("/content/Fase_pruebas"):
        !git clone -q --depth 1 {REPO_URL} /content/Fase_pruebas
    %cd /content/Fase_pruebas
    # opfunu: metadatos de CEC 2022 que importa problems/ (setuptools<82 restaura pkg_resources)
    !pip install -q "setuptools<82" opfunu
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # ejecutado localmente desde notebooks/

RAIZ = os.getcwd()
sys.path.insert(0, RAIZ)
print("Repositorio en", RAIZ)

## 2. Conjunto de algoritmos

`principal+referencia` (12 algoritmos) o `principal` (los 6 de la parrilla
principal). Se fija antes de importar `analisis/`: para cambiarlo, reiniciar la
sesión.

In [ ]:
CONJUNTO = "principal+referencia"      # o "principal"

CARPETA = {"principal": "Resultados_Union",
           "principal+referencia": "Resultados_Union_Referencia"}[CONJUNTO]
ALGS = {"principal": "middleware,pso,de,cmaes,madde,nlshadersp",
        "principal+referencia": "middleware,pso,de,cmaes,madde,nlshadersp,"
                                "ga,jade,clpso,abc,clonalg,optainet"}[CONJUNTO]
os.environ["ANALISIS_ALGORITMOS"] = ALGS

import pandas as pd
from IPython.display import Image, Markdown, display
from analisis.datos import ALGORITMOS, NOMBRE_ALGORITMO, DIMS, MAXFES, cargar_corridas

df = cargar_corridas(f"{CARPETA}/corridas.parquet")      # valida forma, evaluador y checkpoints
print(f"{len(df):,} corridas · k = {len(ALGORITMOS)} algoritmos: "
      + ", ".join(NOMBRE_ALGORITMO[a] for a in ALGORITMOS))

## 3. Generar tablas y figuras

Corre `analisis/main.py` (rankings, Friedman/Shaffer/Wilcoxon, diagramas CD,
convergencia y boxplots). Si la carpeta `analisis/` ya existe se omite; poner
`REGENERAR = True` para forzarlo.

In [ ]:
REGENERAR = False
DIR_ANALISIS = f"{CARPETA}/analisis"
if REGENERAR or not os.path.isfile(f"{DIR_ANALISIS}/figuras/boxplot_rangos_d20.png"):
    !python analisis/main.py --resultados {CARPETA} --algoritmos {CONJUNTO}
else:
    print("Análisis ya generado en", DIR_ANALISIS)

## 4. Tablas resumen

In [ ]:
T = f"{DIR_ANALISIS}/tablas"
nombre = lambda a: NOMBRE_ALGORITMO.get(a, a)

# Rango promedio (1 = mejor) por configuración, algoritmos en columnas
pos = pd.read_csv(f"{T}/posiciones_finales.csv")
rangos = pos.set_index(["dim", "max_fes"])[[f"rango_promedio_{a}" for a in ALGORITMOS]]
rangos.columns = [nombre(a) for a in ALGORITMOS]
display(Markdown("### Rango promedio por configuración (1 = mejor)"))
display(rangos.style.format("{:.2f}").background_gradient(cmap="RdYlGn_r", axis=1))

display(Markdown("### Orden final por configuración"))
display(pos[["dim", "max_fes"] + [f"puesto_{i}" for i in range(1, len(ALGORITMOS) + 1)]])

display(Markdown("### Friedman (α = 0.05)"))
display(pd.read_csv(f"{T}/friedman.csv")[["dim", "max_fes", "estadistico", "p_valor", "rechaza_h0"]])

display(Markdown("### Wilcoxon: propuesto contra cada algoritmo (suma de las 8 configuraciones × 12 funciones)"))
w = pd.read_csv(f"{T}/wilcoxon_bew.csv")
resumen_w = w.groupby(["algoritmo_a", "algoritmo_b"])[["better", "equal", "worse"]].sum()
resumen_w.index = [f"{nombre(a)} vs {nombre(b)}" for a, b in resumen_w.index]
display(resumen_w.sort_values("worse"))

display(Markdown("### % de comparaciones de Shaffer significativas en las que participa el propuesto"))
display(pd.read_csv(f"{T}/shaffer_pct_significativo.csv"))

### Error por función

Media, mediana y desviación del error final de las 51 semillas para una
configuración y función (la mediana más baja en verde).

In [ ]:
import ipywidgets as widgets

sup = pd.read_csv(f"{T}/suplementario_media_mediana_std.csv")
cuart = pd.read_csv(f"{T}/boxplot_cuartiles.csv")

def tabla_funcion(dim, max_fes, funcion):
    s = sup[(sup.dim == dim) & (sup.max_fes == max_fes) & (sup.funcion == funcion)]
    c = cuart[(cuart.dim == dim) & (cuart.max_fes == max_fes) & (cuart.funcion == funcion)]
    t = (s.merge(c[["algoritmo", "q1", "q3", "minimo", "maximo"]], on="algoritmo")
          .set_index("algoritmo").loc[list(ALGORITMOS)]
          [["mean", "median", "std", "minimo", "q1", "q3", "maximo"]])
    t.index = [nombre(a) for a in t.index]
    t.columns = ["media", "mediana", "desv", "mín", "P25", "P75", "máx"]
    return t.sort_values("mediana")

op_fes = [(f"{m:,}".replace(",", "."), m) for m in MAXFES]
widgets.interact(
    lambda dim, max_fes, funcion: display(
        tabla_funcion(dim, max_fes, funcion).style.format("{:.3e}")
        .highlight_min(subset=["mediana"], color="#b7e1b7")),
    dim=widgets.Dropdown(options=list(DIMS), description="D"),
    max_fes=widgets.Dropdown(options=op_fes, description="MaxFES"),
    funcion=widgets.IntSlider(min=1, max=12, value=1, description="Función"),
);

## 5. Figuras

Elegir el tipo de figura y la configuración. Los *paneles* muestran las 12
funciones; las figuras *por función* usan el selector de función.

In [ ]:
from pathlib import Path
F = Path(DIR_ANALISIS) / "figuras"

TIPOS = {
    "Boxplot del error final (panel)": lambda d, m, f: F / f"boxplot_d{d}_fes{m}.png",
    "Boxplot del error final (por función)": lambda d, m, f: F / "boxplot" / f"d{d}_fes{m}" / f"f{f:02d}.png",
    "Convergencia (panel)": lambda d, m, f: F / f"convergencia_d{d}_fes{m}.png",
    "Convergencia (por función)": lambda d, m, f: F / "convergencia" / f"d{d}_fes{m}" / f"f{f:02d}.png",
    "Diagrama CD de Nemenyi": lambda d, m, f: F / f"cd_d{d}_fes{m}.png",
    "Boxplot de rangos (los 4 MaxFES)": lambda d, m, f: F / f"boxplot_rangos_d{d}.png",
    "Barras de rango promedio (los 4 MaxFES)": lambda d, m, f: F / f"barras_rangos_d{d}.png",
}

def ver_figura(tipo, dim, max_fes, funcion):
    ruta = TIPOS[tipo](dim, max_fes, funcion)
    if ruta.exists():
        display(Image(filename=str(ruta)))
    else:
        print("No existe", ruta, "— ¿se generó el análisis (sección 3)?")

widgets.interact(
    ver_figura,
    tipo=widgets.Dropdown(options=list(TIPOS), description="Figura",
                          layout=widgets.Layout(width="420px")),
    dim=widgets.Dropdown(options=list(DIMS), description="D"),
    max_fes=widgets.Dropdown(options=op_fes, description="MaxFES"),
    funcion=widgets.IntSlider(min=1, max=12, value=1, description="Función"),
);

### Boxplot a medida

Boxplot del error final directamente desde los datos, para una función y un
subconjunto de algoritmos (útil para comparar pocos algoritmos a la vez).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def boxplot_a_medida(dim, max_fes, funcion, algoritmos, log):
    algoritmos = [a for a in ALGORITMOS if a in algoritmos]
    if not algoritmos:
        print("Elegir al menos un algoritmo."); return
    sub = df[(df.dim == dim) & (df.max_fes == max_fes) & (df.funcion == funcion)]
    datos = [np.maximum(sub.loc[sub.algoritmo == a, "error"].to_numpy(), 1e-8) for a in algoritmos]
    fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(algoritmos) + 2), 4.5))
    ax.boxplot(datos, tick_labels=[nombre(a) for a in algoritmos], patch_artist=True,
               medianprops=dict(color="black"), boxprops=dict(facecolor="#9cc3ee"))
    if log:
        ax.set_yscale("log")
    ax.set_ylabel("Error final f − f(x*)")
    ax.set_title(f"F{funcion} — D={dim}, MaxFES={max_fes:,}".replace(",", ".") + " (51 semillas)")
    ax.tick_params(axis="x", rotation=60)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

widgets.interact(
    boxplot_a_medida,
    dim=widgets.Dropdown(options=list(DIMS), description="D"),
    max_fes=widgets.Dropdown(options=op_fes, description="MaxFES"),
    funcion=widgets.IntSlider(min=1, max=12, value=1, description="Función"),
    algoritmos=widgets.SelectMultiple(options=[(nombre(a), a) for a in ALGORITMOS],
                                      value=tuple(ALGORITMOS), rows=12, description="Algoritmos"),
    log=widgets.Checkbox(value=True, description="Escala log"),
);

## 6. Descargar el análisis completo

In [ ]:
# Comprime tablas y figuras en un .zip y lo descarga (solo en Colab).
import shutil
zip_ruta = shutil.make_archive(f"analisis_{CONJUNTO.replace('+', '_')}", "zip", DIR_ANALISIS)
print("Generado", zip_ruta)
if EN_COLAB:
    from google.colab import files
    files.download(zip_ruta)

## 7. Cómo leer cada tabla y figura

In [ ]:
display(Markdown(Path(DIR_ANALISIS, "LEEME.md").read_text(encoding="utf-8")))